# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshpnsb/ML-INTERNSHIP/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** CTR / Engagement Opportunity Scoring

This notebook defines the data contract for predicting which visible pages under-capture clicks
or engagement — the pages a reviewer should look at first. It covers the contract (five plain-words
answers), three verification queries on a mid-panel month (2026-03), five engineered features with
availability notes, and a deliberate leakage experiment on real warehouse data.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill
> this assignment names on its card.

## 0. Setup — connect to the warehouse release

In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
T = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in T.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


---
## 1. The Contract — five plain-words answers

### 1) What does one row mean?

One row = **one content item on one day for one client**. The grain is `(report_date, client_hash_id, content_hash_id)` from `fact_content_daily_performance`. That is the atomic event: a single page's search and engagement signals for a single calendar day.

### 2) Which tables?

- `fact_content_daily_performance` — daily GSC + GA4 signals (the fact table).
- `dim_content` — content metadata (content type, keyword context) for joins and feature enrichment.
- `dim_clients` — client-level access profile and history start dates, used to check panel coverage.

### 3) Which time window?

- **Feature window:** the 30 days ending on the last day of the month being analyzed (e.g. for `month=2026-03`, features span 2026-03-02 to 2026-03-31). This is the "prior" window we observe.
- **Label window:** the 30 days AFTER the feature window (e.g. April 2026 for a March feature window). We predict a future outcome from past signals.
- For this notebook's queries we work with `month=2026-03` as the feature month. The label month (April) is only used in the leakage experiment.

### 4) What do we predict or rank? (Label / proxy)

**Label:** `CTR decline` — a binary label: 1 if the page's average CTR in the feature window (March) drops by more than 20% relative to the next 30-day window (April), 0 otherwise. We rank pages by the probability of this decline so a reviewer can prioritize which pages to investigate.

For the three verification queries we use a **proxy label** computed from the same month: `low_ctr_residual` = 1 if the page's CTR is below the position-tier median by more than 1 percentage point. This proxy is available at decision time (it uses only current-month data) and lets us sanity-check feature quality without peeking at the future.

### 5) What do we deliberately exclude?

**`gsc_avg_position`** is excluded as a direct feature. Position is a strong predictor of CTR, but it is also an output of Google's ranking algorithm — using it directly makes the model learn "pages ranked higher get more clicks," which is tautological. Instead we use position-tier indicators (top_3, page_1, etc.) as coarse context, and we compute a position-adjusted CTR residual as the label. This keeps the model focused on *why* a page under-performs its tier, not just *which tier it sits in*.

In [ ]:
# Quick sanity: confirm the fact table is partitioned by month and see what months exist.
con.sql(f"""
    SELECT month, COUNT(*) AS rows
    FROM {T['fact_daily']}
    GROUP BY month
    ORDER BY month
""").df()


---
## 2. Three verification queries (month=2026-03)

Every claim in the contract gets a query. We use `month='2026-03'` — a mid-panel month that is cheap to scan and avoids peeking at the final (June) test month.

### Query 1 — Grain check: one row really is `(report_date, client_hash_id, content_hash_id)`

If the grain holds, grouping by those three columns and counting should return exactly 1 per group — zero duplicates.

In [ ]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS cnt
    FROM {T['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

if grain_check.empty:
    print('GRAIN OK: zero duplicate rows for (report_date, client_hash_id, content_hash_id) in month=2026-03')
else:
    print('GRAIN VIOLATION — duplicates found:')
    display(grain_check)


### Query 2 — Row count and date span for month=2026-03

How many rows does this slice contain, and what is the earliest and latest `report_date`?

In [ ]:
slice_stats = con.sql(f"""
    SELECT
        COUNT(*)                    AS total_rows,
        COUNT(DISTINCT client_hash_id)  AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date)            AS earliest_date,
        MAX(report_date)            AS latest_date
    FROM {T['fact_daily']}
    WHERE month = '2026-03'
""").df()

display(slice_stats)
print(f"Date span: {slice_stats['earliest_date'].iloc[0]} to {slice_stats['latest_date'].iloc[0]}")


### Query 3 — Availability: filter with `IS TRUE` and show how many rows survive

The `ga4_data_available` flag is the critical filter. Many rows before a client's GA4 start date are
zero-filled — those zeros mean "not measured," not "no engagement." Writing `= FALSE` or
`NOT flag` gives wrong counts because NULL is neither TRUE nor FALSE. The correct filter is
`IS TRUE`.

In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*)                                    AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)  AS ga4_true,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_not_true,
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END)          AS has_impressions,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS ga4_pct,
        ROUND(100.0 * SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) / COUNT(*), 1)        AS imp_pct
    FROM {T['fact_daily']}
    WHERE month = '2026-03'
""").df()

display(availability)

ga4_true = availability['ga4_true'].iloc[0]
total    = availability['total_rows'].iloc[0]
print(f"\nAfter IS TRUE filter: {ga4_true:,} rows survive out of {total:,} ({availability['ga4_pct'].iloc[0]}%)")
print(f"Note: the {availability['ga4_not_true'].iloc[0]:,} non-TRUE rows include both FALSE and NULL — a blind '= FALSE' would miss the NULLs.")


---
## 3. Five features — built from month=2026-03

Max five features. Each gets one line: *"knowable at the decision moment because…"*

We aggregate daily rows into one row per content item for the feature month, then compute:

| # | Feature | Line |
|---|---------|------|
| 1 | `avg_ctr_30d` | Mean daily CTR over the 30-day feature window — knowable because it uses only GSC data from the same month. |
| 2 | `avg_position_30d` | Mean daily average position over the window — knowable from GSC for the same month. |
| 3 | `impressions_total_30d` | Total GSC impressions in the window — a volume proxy, knowable at decision time. |
| 4 | `session_rate_30d` | GA4 sessions / impressions ratio over the window — knowable from GA4 for the same month (filtered by IS TRUE). |
| 5 | `impression_volatility_30d` | Coefficient of variation of daily impressions — measures stability, knowable from daily-level data within the feature window. |

In [ ]:
features_df = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,

        -- Feature 1: avg CTR over the 30-day window
        AVG(CASE WHEN f.gsc_impressions > 0
                  THEN f.gsc_clicks::DOUBLE / f.gsc_impressions
             END) AS avg_ctr_30d,

        -- Feature 2: avg position over the window
        AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position END) AS avg_position_30d,

        -- Feature 3: total impressions (volume proxy)
        SUM(f.gsc_impressions) AS impressions_total_30d,

        -- Feature 4: session rate (sessions / impressions) — only for GA4-available rows
        SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.ga4_sessions ELSE 0 END)::DOUBLE
        / NULLIF(SUM(f.gsc_impressions), 0) AS session_rate_30d,

        -- Feature 5: impression volatility (stddev / mean of daily impressions)
        STDDEV(f.gsc_impressions)::DOUBLE
        / NULLIF(AVG(f.gsc_impressions), 0) AS impression_volatility_30d

    FROM {T['fact_daily']} f
    WHERE f.month = '2026-03'
      AND f.gsc_impressions > 0
    GROUP BY f.content_hash_id, f.client_hash_id
    HAVING COUNT(*) >= 5   -- require at least 5 days of data in the window
""").df()

print(f"Feature frame: {len(features_df):,} content items from month=2026-03")
display(features_df.describe().round(4))


### Feature availability summary

| Feature | Available when? | Why it doesn't leak |
|---------|----------------|---------------------|
| `avg_ctr_30d` | Known at end of feature month | Computed only from GSC clicks and impressions within the 30-day window being analyzed. |
| `avg_position_30d` | Known at end of feature month | GSC average position within the same 30-day window. |
| `impressions_total_30d` | Known at end of feature month | Sum of GSC impressions within the window. |
| `session_rate_30d` | Known at end of feature month (requires GA4 IS TRUE) | GA4 sessions / impressions, both measured within the window. |
| `impression_volatility_30d` | Known at end of feature month | Stddev / mean of daily impressions within the window — uses only daily-level data from the same month. |

In [ ]:
# Show the first few rows of the feature frame
display(features_df.head(10))


---
## 4. The Trap — deliberate leakage experiment

We add ONE label-derived column on purpose, watch the score jump toward perfect, then delete it
and keep the honest number. This is the leakage lesson from notebook 02, performed on real
warehouse data.

**The label-derived column:** `clicks_next_30d` — the number of GSC clicks in the month
AFTER our feature window (April 2026). This is the answer in disguise: a model given this column
can trivially predict any April outcome. The score will look amazing and mean nothing.

**The proxy label we use for this demo:** `low_ctr` = 1 if the page's average CTR in March is
below 0.5% (a common threshold for "under-performing"). We predict this from our five honest
features, then from five features + the leaked column.

In [ ]:
# Build the proxy label and attach future-window clicks (the leak)
leak_df = con.sql(f"""
    WITH march_features AS (
        SELECT
            content_hash_id,
            client_hash_id,
            AVG(CASE WHEN gsc_impressions > 0
                      THEN gsc_clicks::DOUBLE / gsc_impressions END) AS avg_ctr_30d,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_30d,
            SUM(gsc_impressions) AS impressions_total_30d,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END)::DOUBLE
                / NULLIF(SUM(gsc_impressions), 0) AS session_rate_30d,
            STDDEV(gsc_impressions)::DOUBLE / NULLIF(AVG(gsc_impressions), 0) AS impression_volatility_30d
        FROM {T['fact_daily']}
        WHERE month = '2026-03' AND gsc_impressions > 0
        GROUP BY content_hash_id, client_hash_id
        HAVING COUNT(*) >= 5
    ),
    april_leak AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_clicks) AS clicks_next_30d  -- THIS IS THE LEAK
        FROM {T['fact_daily']}
        WHERE month = '2026-04'
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        m.*,
        COALESCE(a.clicks_next_30d, 0) AS clicks_next_30d,
        CASE WHEN m.avg_ctr_30d < 0.005 THEN 1 ELSE 0 END AS low_ctr_label
    FROM march_features m
    LEFT JOIN april_leak a
        ON m.content_hash_id = a.content_hash_id
       AND m.client_hash_id = a.client_hash_id
""").df()

print(f"Leakage frame: {len(leak_df):,} rows")
print(f"low_ctr label rate: {leak_df['low_ctr_label'].mean():.3f}")
display(leak_df.head())


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

honest_features = ['avg_ctr_30d', 'avg_position_30d', 'impressions_total_30d',
                   'session_rate_30d', 'impression_volatility_30d']

# --- Honest model (no leak) ---
X_honest = leak_df[honest_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = leak_df['low_ctr_label'].values

model_honest = LogisticRegression(max_iter=500, random_state=42)
model_honest.fit(X_honest, y)
honest_proba = model_honest.predict_proba(X_honest)[:, 1]

# --- Leaky model (with clicks_next_30d) ---
leaky_features = honest_features + ['clicks_next_30d']
X_leaky = leak_df[leaky_features].replace([np.inf, -np.inf], np.nan).fillna(0)

model_leaky = LogisticRegression(max_iter=500, random_state=42)
model_leaky.fit(X_leaky, y)
leaky_proba = model_leaky.predict_proba(X_leaky)[:, 1]

# --- Score both with precision@20 ---
def precision_at_k(proba, labels, k):
    order = np.argsort(-proba)
    return labels[order[:k]].mean()

print("=== DELIBERATE LEAKAGE EXPERIMENT ===")
print(f"Honest model  (5 features)             — AUC: {roc_auc_score(y, honest_proba):.3f}  |  Precision@20: {precision_at_k(honest_proba, y, 20):.3f}")
print(f"Leaky model   (5 features + next month) — AUC: {roc_auc_score(y, leaky_proba):.3f}  |  Precision@20: {precision_at_k(leaky_proba, y, 20):.3f}")
print(f"\nThe leaky model looks amazing. It should — it was given the answer.")
print(f"We delete it and keep the honest number.")


In [ ]:
# The honest result — this is what we keep

print("=== HONEST RESULT (no leakage) ===")
print(f"Features: {honest_features}")
print(f"Label:    low_ctr (avg CTR < 0.5% in March 2026)")
print(f"AUC:      {roc_auc_score(y, honest_proba):.3f}")
print(f"P@20:     {precision_at_k(honest_proba, y, 20):.3f}")
print(f"P@50:     {precision_at_k(honest_proba, y, 50):.3f}")
print(f"\nThis number is the honest baseline we will try to beat in later weeks.")


---
## 5. One named limitation of this slice

**Unbalanced panel depth.** Not all clients have 30 days of data in March 2026. Some clients
started tracking after March began, and some have gaps where `ga4_data_available` is not TRUE.
This means our feature frame is biased toward clients with complete March coverage — we are
measuring "pages with enough data" not "all pages." The `HAVING COUNT(*) >= 5` filter helps
but does not fix the underlying imbalance. A production model should use per-client windows
(check `dim_clients.gsc_data_start` and `ga4_data_start`) rather than a global calendar window.
We will address this in the signal audit (week 4) and validation (week 6) notebooks.

**Secondary limitation:** the proxy label (`low_ctr < 0.5%`) is a threshold-based bucket, not a
future outcome. The real label (CTR decline in April) requires the full fact table scan and will
be defined in the modeling notebook. This proxy is for contract verification only.

---
## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.